# Part 1: Prepare Transcript Inputs

This notebook only prepares and validates the inputs. It does not train a model.

The output contains the combined official train/dev/test PHQ-8 labels and three transcript versions for every shared participant: Official, Raw Whisper, and Trimmed Whisper.

## 1. Imports and Paths

In [1]:
from pathlib import Path
import re

import pandas as pd

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebook" else Path.cwd()
LABEL_PATHS = {
    "train": PROJECT_ROOT / "data" / "train_split.csv",
    "dev": PROJECT_ROOT / "data" / "dev_split.csv",
    "test": PROJECT_ROOT / "data" / "test_split.csv",
}
OFFICIAL_DIR = PROJECT_ROOT / "data" / "transcript"
RAW_DIR = PROJECT_ROOT / "outputs" / "asr" / "whisper_base"
TRIMMED_DIR = PROJECT_ROOT / "database" / "formatted" / "transcript"
INPUT_DIR = PROJECT_ROOT / "input"
COMBINED_LABEL_PATH = INPUT_DIR / "combined_phq8_labels.csv"
OUTPUT_PATH = INPUT_DIR / "transcript_model_input.csv"
INPUT_DIR.mkdir(parents=True, exist_ok=True)

required_paths = [*LABEL_PATHS.values(), OFFICIAL_DIR, RAW_DIR, TRIMMED_DIR]
for path in required_paths:
    if not path.exists():
        raise FileNotFoundError(path)

print(f"Project root: {PROJECT_ROOT}")

Project root: e:\Project\daic-woz-audio-analysis


## 2. Prepare PHQ-8 Labels

Classes: 0-4 None, 5-9 Mild, 10-14 Moderate, 15-19 Moderately severe, and 20-24 Severe.

In [2]:
SEVERITY_NAMES = {
    0: "None",
    1: "Mild",
    2: "Moderate",
    3: "Moderately severe",
    4: "Severe",
}

label_frames = []
for official_split, path in LABEL_PATHS.items():
    frame = pd.read_csv(path)
    required_columns = {"Participant_ID", "PHQ_Score"}
    missing_columns = required_columns - set(frame.columns)
    if missing_columns:
        raise ValueError(f"Missing columns in {path.name}: {sorted(missing_columns)}")

    frame = frame[["Participant_ID", "PHQ_Score"]].copy()
    frame["Official_Split"] = official_split
    label_frames.append(frame)

labels = pd.concat(label_frames, ignore_index=True)
labels = labels.rename(columns={"PHQ_Score": "PHQ_8Total"})
labels["Participant_ID"] = labels["Participant_ID"].astype(int)
labels["PHQ_8Total"] = pd.to_numeric(labels["PHQ_8Total"], errors="raise")

assert labels["Participant_ID"].is_unique, "Participant appears in multiple official splits"
assert labels["PHQ_8Total"].between(0, 24).all()

labels["PHQ8_Severity"] = pd.cut(
    labels["PHQ_8Total"],
    bins=[-1, 4, 9, 14, 19, 24],
    labels=[0, 1, 2, 3, 4],
).astype(int)
labels["PHQ8_Severity_Name"] = labels["PHQ8_Severity"].map(SEVERITY_NAMES)
labels.to_csv(COMBINED_LABEL_PATH, index=False)

print(f"Participants with combined official labels: {len(labels)}")
print(f"Saved combined labels to: {COMBINED_LABEL_PATH}")
display(labels["Official_Split"].value_counts().reindex(LABEL_PATHS).rename("participants"))
display(
    labels["PHQ8_Severity_Name"]
    .value_counts()
    .reindex(SEVERITY_NAMES.values(), fill_value=0)
)

Participants with combined official labels: 275
Saved combined labels to: e:\Project\daic-woz-audio-analysis\input\combined_phq8_labels.csv


Official_Split
train    163
dev       56
test      56
Name: participants, dtype: int64

PHQ8_Severity_Name
None                 122
Mild                  67
Moderate              43
Moderately severe     33
Severe                10
Name: count, dtype: int64

## 3. Load Transcript Text

Rows are joined in their existing order. Empty cells are ignored and repeated whitespace is normalized.

In [3]:
def participant_id_from_path(path: Path) -> int:
    return int(path.stem.split("_")[0])


def load_corpus(directory: Path, pattern: str, prefix: str) -> pd.DataFrame:
    rows = []

    for path in sorted(directory.glob(pattern)):
        transcript = pd.read_csv(path)
        if "Text" not in transcript.columns:
            raise ValueError(f"Missing Text column: {path}")

        segments = transcript["Text"].dropna().astype(str)
        text = re.sub(r"\s+", " ", " ".join(segments)).strip()
        rows.append(
            {
                "Participant_ID": participant_id_from_path(path),
                f"{prefix}_text": text,
                f"{prefix}_segments": len(transcript),
                f"{prefix}_words": len(text.split()),
                f"{prefix}_path": str(path),
            }
        )

    result = pd.DataFrame(rows)
    if result.empty:
        raise ValueError(f"No transcript files found in {directory}")
    if not result["Participant_ID"].is_unique:
        raise ValueError(f"Duplicate participant transcript in {directory}")
    return result


official = load_corpus(OFFICIAL_DIR, "*_TRANSCRIPT.csv", "official")
raw_whisper = load_corpus(RAW_DIR, "*_whisper.csv", "raw_whisper")
trimmed_whisper = load_corpus(TRIMMED_DIR, "*_whisper.csv", "trimmed_whisper")

display(
    pd.DataFrame(
        {
            "source": ["PHQ-8 labels", "Official", "Raw Whisper", "Trimmed Whisper"],
            "participants": [len(labels), len(official), len(raw_whisper), len(trimmed_whisper)],
        }
    )
)

,source,participants
0,PHQ-8 labels,275
1,Official,275
2,Raw Whisper,275
3,Trimmed Whisper,202


## 4. Keep the Same Participants

The inner joins keep only participants who have a label and all three transcript versions.

In [4]:
dataset = (
    labels
    .merge(official, on="Participant_ID", validate="one_to_one")
    .merge(raw_whisper, on="Participant_ID", validate="one_to_one")
    .merge(trimmed_whisper, on="Participant_ID", validate="one_to_one")
    .sort_values("Participant_ID")
    .reset_index(drop=True)
)

print(f"Participants available for a fair comparison: {len(dataset)}")
display(
    dataset["PHQ8_Severity_Name"]
    .value_counts()
    .reindex(SEVERITY_NAMES.values(), fill_value=0)
)

Participants available for a fair comparison: 202


PHQ8_Severity_Name
None                 87
Mild                 49
Moderate             32
Moderately severe    25
Severe                9
Name: count, dtype: int64

## 5. Validate and Inspect the Prepared Inputs

In [5]:
TEXT_COLUMNS = ["official_text", "raw_whisper_text", "trimmed_whisper_text"]
WORD_COLUMNS = ["official_words", "raw_whisper_words", "trimmed_whisper_words"]

empty_counts = dataset[TEXT_COLUMNS].apply(lambda column: column.str.strip().eq("").sum())
assert empty_counts.eq(0).all(), f"Empty transcript inputs:\n{empty_counts}"

dataset["trimmed_fraction_of_raw"] = (
    dataset["trimmed_whisper_words"]
    / dataset["raw_whisper_words"].replace(0, pd.NA)
)

display(dataset[WORD_COLUMNS + ["trimmed_fraction_of_raw"]].describe().round(2))
print(f"Empty inputs: {int(empty_counts.sum())}")
print(
    "Trimmed longer than raw: ",
    int((dataset["trimmed_whisper_words"] > dataset["raw_whisper_words"]).sum()),
)
display(
    dataset[
        ["Participant_ID", "PHQ_8Total", "PHQ8_Severity_Name"] + WORD_COLUMNS
    ].head(10)
)

,official_words,raw_whisper_words,trimmed_whisper_words,trimmed_fraction_of_raw
count,202.00,202.00,202.00,202.00
mean,1313.28,1685.40,1616.61,0.95
std,777.91,763.98,765.47,0.07
min,170.00,506.00,412.00,0.52
25%,703.00,1110.25,1047.75,0.94
50%,1184.50,1518.00,1465.50,0.97
75%,1702.00,2055.25,1976.00,0.99
max,4269.00,4694.00,4644.00,1.00


Empty inputs: 0
Trimmed longer than raw:  0


,Participant_ID,PHQ_8Total,PHQ8_Severity_Name,official_words,raw_whisper_words,trimmed_whisper_words
0,300,2,None,322,797,713
1,301,3,None,1399,1624,1569
2,302,4,None,609,1006,927
3,303,0,None,1916,2385,2321
4,304,6,Mild,992,1511,1450
5,305,7,Mild,3132,3337,3255
6,306,0,None,1817,1950,1870
7,307,4,None,2631,3026,2946
8,308,22,Severe,1073,1251,1176
9,309,15,Moderately severe,651,807,777


## 6. Save the Input Table

In [6]:
dataset.to_csv(OUTPUT_PATH, index=False)
print(f"Saved {len(dataset)} rows to: {OUTPUT_PATH}")
print(f"Columns: {len(dataset.columns)}")

Saved 202 rows to: e:\Project\daic-woz-audio-analysis\input\transcript_model_input.csv
Columns: 18


## Stop Here

Review the prepared input before adding any text encoder or classification model.